# JEV — pourquoi la définition du `state` compte

### Cas : « Je veux laver ma voiture, le carwash est à 50 mètres »

Ce notebook montre **le même problème en deux versions** avec `typesafe/jev-1.13` via l’API **OpenRouter Decisions** :

1. **State mal défini** : l’objectif réel est implicite. JEV peut raisonnablement interpréter « 50 m » comme une question de déplacement personnel et choisir **marcher**.
2. **State bien défini** : on explicite que **la voiture doit arriver au carwash pour être lavée** et ce que signifient exactement les actions.

> Idée clé : le typage contraint la forme de la décision, mais il ne répare pas un état du monde incomplet ou ambigu.

## 1. Configuration

Le notebook demande votre clé OpenRouter sans l’afficher.

In [ ]:
import getpass, json, requests, pandas as pd

OPENROUTER_API_KEY = getpass.getpass("OPENROUTER_API_KEY: ")
MODEL = "typesafe/jev-1.13"
URL = "https://openrouter.ai/api/alpha/decisions"

def ask_jev(state, questions):
    r = requests.post(
        URL,
        headers={
            "Authorization": f"Bearer {OPENROUTER_API_KEY}",
            "Content-Type": "application/json",
        },
        json={"model": MODEL, "state": state, "questions": questions},
        timeout=60,
    )
    r.raise_for_status()
    return r.json()

def show_choice(result, question_id="transport"):
    a = result["answers"][question_id]
    print("Décision :", a["choice"])
    print("Confiance:", a.get("confidence"))
    print("Probabilités:")
    for k, v in a["probabilities"].items():
        print(f"  {k:>6}: {v:.1%}")
    print("\nUsage:", result.get("usage", {}))

# Expérience A — state mal défini

Ici, on fournit seulement :

> « Je veux laver ma voiture. Le carwash est à 50 mètres. »

Le problème est que **la contrainte essentielle n’est pas représentée** : pour atteindre l’objectif, la voiture doit elle-même se trouver au carwash.

La question contient aussi des justifications concurrentes : la faible distance pousse vers `walk`, tandis que l’objectif « laver la voiture » pousse vers `drive`.

In [ ]:
state_bad = {
    "situation": "I want to wash my car. The car wash is 50 meters away."
}

questions_bad = {
    "transport": {
        "type": "choice",
        "instructions": "What should I do?",
        "criteria": {
            "walk": "Walk, as it is just 50 meters away.",
            "drive": "Drive, as I have to wash my car."
        }
    }
}

print(json.dumps({"state": state_bad, "questions": questions_bad}, indent=2))

In [ ]:
result_bad = ask_jev(state_bad, questions_bad)
show_choice(result_bad)

### Pourquoi ce résultat peut être mauvais

Si JEV choisit `walk`, ce n’est pas nécessairement un échec de classification. Le `state` ne dit jamais explicitement que :

- le lavage concerne **cette voiture** ;
- la voiture est actuellement avec moi ;
- **marcher** signifie aller au carwash **sans déplacer la voiture** ;
- l’objectif n’est satisfait que si **la voiture arrive au carwash**.

Autrement dit, plusieurs mondes compatibles avec le `state` sont possibles.

# Expérience B — state bien défini

On rend maintenant explicites **l’objectif, les objets pertinents et la conséquence des actions**.

La distance reste exactement la même : **50 mètres**. Ce n’est donc pas l’information factuelle qui change, mais la représentation du problème.

In [ ]:
state_good = {
    "goal": "Wash my car at the car wash",
    "distance_to_car_wash_m": 50,
    "current_location_of_car": "with me",
    "goal_requirement": "My car must be physically present at the car wash to be washed"
}

questions_good = {
    "transport": {
        "type": "choice",
        "instructions": (
            "Which action satisfies the stated goal? "
            "Choose based on whether the car itself reaches the car wash."
        ),
        "criteria": {
            "walk": "I walk to the car wash without moving my car.",
            "drive": "I drive my car to the car wash, so the car arrives there."
        }
    }
}

print(json.dumps({"state": state_good, "questions": questions_good}, indent=2))

In [ ]:
result_good = ask_jev(state_good, questions_good)
show_choice(result_good)

# Comparaison

On affiche les distributions côte à côte. L’intérêt de JEV n’est pas seulement le label choisi : **la distribution de probabilités rend visible l’incertitude**.

In [ ]:
def probs(result):
    return result["answers"]["transport"]["probabilities"]

comparison = pd.DataFrame({
    "State mal défini": probs(result_bad),
    "State bien défini": probs(result_good)
})
comparison.index.name = "action"
comparison.style.format("{:.1%}")

## Ce que démontre l’expérience

**JEV décide sur le monde que vous lui décrivez, pas sur le monde que vous aviez en tête.**

Le premier contrat est syntaxiquement valide : JEV doit choisir parmi `walk` et `drive`. Mais le problème sémantique reste sous-spécifié.

Le second contrat encode les invariants utiles au raisonnement :

`objectif → voiture nécessaire au carwash → marcher ne déplace pas la voiture → conduire satisfait l’objectif`

### À retenir

> **Type-safe ≠ world-safe.**  
> Un output peut être parfaitement typé tout en étant une mauvaise décision si le `state` ne contient pas les contraintes déterminantes.

Dans une application réelle, on peut aller encore plus loin : garder JEV pour les **jugements sémantiques bornés**, et laisser au code déterministe les contraintes qui sont déjà connues avec certitude.